# ⚽ Football Match Outcome Prediction

## Author
Marina Pérez Sáiz

## Status
🚧 Work in Progress

## Objective

Develop an end-to-end machine learning pipeline capable of predicting football match outcomes using historical data.

The project currently focuses on building the data collection pipeline and establishing a first predictive baseline before incorporating more advanced football-specific features.

# 1. Import Libraries

In [4]:
import requests
import sqlite3
from datetime import datetime
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import os
from dotenv import load_dotenv

# 2. Create Local Database

This section creates a local SQLite database used to store historical football matches.

In [ ]:
# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('soccer_data.db')

# Create a cursor object to execute SQL commands
cursor = conn.cursor()

# Create a table to store match data if it doesn't already exist
cursor.execute('''
    CREATE TABLE IF NOT EXISTS matches (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        home_team TEXT,
        away_team TEXT,
        home_score INTEGER,
        away_score INTEGER,
        match_date TEXT,
        UNIQUE(match_date, home_team, away_team)  -- Prevent duplicates
    )
''')

# Commit the transaction to save changes
conn.commit()
     
# Close the database connection when done (optional at this point, but good practice)
conn.close()

print("Database and table created successfully.")

# Database setup
conn = sqlite3.connect('soccer_data.db')
cursor = conn.cursor()

# Load API key from environment variables
load_dotenv()
matches_url = "https://api.football-data.org/v4/competitions/PD/matches"
api_key = os.getenv("FOOTBALL_API_KEY")
headers = {"X-Auth-Token": api_key}

Database and table created successfully.


# 3. Retrieve Historical Match Data

Historical matches are collected through the Football-Data.org API.

Only completed matches that are not already stored in the database are inserted.

In [ ]:
def fetch_data_with_limits(matches_url, headers):
    try:
        response = requests.get(matches_url, headers=headers)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error: Status code {response.status_code}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return None

def get_existing_matches(cursor):
    cursor.execute('SELECT match_date, home_team, away_team FROM matches')
    return set(cursor.fetchall())  # Using a set for faster lookups

def insert_new_matches_to_db(matches, cursor):
    existing_matches = get_existing_matches(cursor)
    today = datetime.utcnow().date()  # Get today's date in UTC

    for match in matches:
        match_date = match['utcDate']  # Assume UTC date format from API
        home_team = match['homeTeam']['name']
        away_team = match['awayTeam']['name']
        
        match_date_obj = datetime.fromisoformat(match_date[:-1]).date()  # Convert to date object
        
        # Check if match date is before today and not in existing matches
        if match_date_obj < today and (match_date, home_team, away_team) not in existing_matches:
            home_score = match['score']['fullTime']['home']
            away_score = match['score']['fullTime']['away']
            
            try:
                cursor.execute('''
                    INSERT INTO matches (match_date, home_team, away_team, home_score, away_score)
                    VALUES (?, ?, ?, ?, ?)
                ''', (match_date, home_team, away_team, home_score, away_score))
                print(f"Inserted match: {home_team} vs {away_team} on {match_date}")
            except sqlite3.IntegrityError:
                # Skip the insert if it would create a duplicate entry
                print(f"Match on {match_date} between {home_team} and {away_team} already exists.")

# Example usage
matches_data = fetch_data_with_limits(matches_url, headers)
if matches_data and 'matches' in matches_data:
    insert_new_matches_to_db(matches_data['matches'], cursor)
    conn.commit()  # Save changes to the database
else:
    print("Failed to fetch or no matches available.")

# Connect to the SQLite database
conn = sqlite3.connect('soccer_data.db')
cursor = conn.cursor()

# To display the data in the database:
cursor.execute('SELECT * FROM matches')
rows =cursor.fetchall()

if rows:
       for row in rows:
            print(f"ID: {row[0]}, Home Team: {row[1]}, Away Team: {row[2]}, Home Score: {row[3]}, Away Score: {row[4]}, Match Date: {row[5]}")
else:
     print("No match data found.")

# Close the database connection when done (optional at this point, but good practice)
conn.close()

ID: 1, Home Team: Athletic Club, Away Team: Getafe CF, Home Score: 1, Away Score: 1, Match Date: 2024-08-15T17:00:00Z
ID: 2, Home Team: Real Betis Balompié, Away Team: Girona FC, Home Score: 1, Away Score: 1, Match Date: 2024-08-15T19:30:00Z
ID: 3, Home Team: RC Celta de Vigo, Away Team: Deportivo Alavés, Home Score: 2, Away Score: 1, Match Date: 2024-08-16T17:00:00Z
ID: 4, Home Team: UD Las Palmas, Away Team: Sevilla FC, Home Score: 2, Away Score: 2, Match Date: 2024-08-16T19:30:00Z
ID: 5, Home Team: CA Osasuna, Away Team: CD Leganés, Home Score: 1, Away Score: 1, Match Date: 2024-08-17T17:00:00Z
ID: 6, Home Team: Valencia CF, Away Team: FC Barcelona, Home Score: 1, Away Score: 2, Match Date: 2024-08-17T19:30:00Z
ID: 7, Home Team: Real Sociedad de Fútbol, Away Team: Rayo Vallecano de Madrid, Home Score: 1, Away Score: 2, Match Date: 2024-08-18T17:00:00Z
ID: 8, Home Team: RCD Mallorca, Away Team: Real Madrid CF, Home Score: 1, Away Score: 1, Match Date: 2024-08-18T19:30:00Z
ID: 9, Home

C:\Users\marina.perez\AppData\Local\Temp\ipykernel_11540\703767298.py:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().date()  # Get today's date in UTC


# 4. Load Dataset

In [9]:
# Connect to the SQLite database
conn = sqlite3.connect('soccer_data.db')

# Define your SQL query to fetch the data you want
query = 'SELECT * FROM matches'  # You can adjust this to your needs

# Load the data into a DataFrame
matches_df = pd.read_sql_query(query, conn)

# Close the database connection
conn.close()

## Explore the Dataset

In [10]:
matches_df.head()

,id,home_team,away_team,home_score,away_score,match_date
0,1,Athletic Club,Getafe CF,1,1,2024-08-15T17:00:00Z
1,2,Real Betis Balompié,Girona FC,1,1,2024-08-15T19:30:00Z
2,3,RC Celta de Vigo,Deportivo Alavés,2,1,2024-08-16T17:00:00Z
3,4,UD Las Palmas,Sevilla FC,2,2,2024-08-16T19:30:00Z
4,5,CA Osasuna,CD Leganés,1,1,2024-08-17T17:00:00Z


Dataset information

In [11]:
matches_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 260 entries, 0 to 259
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          260 non-null    int64
 1   home_team   260 non-null    str  
 2   away_team   260 non-null    str  
 3   home_score  260 non-null    int64
 4   away_score  260 non-null    int64
 5   match_date  260 non-null    str  
dtypes: int64(3), str(3)
memory usage: 12.3 KB


Statistical summary

In [12]:
matches_df.describe()

,id,home_score,away_score
count,260.000000,260.000000,260.000000
mean,130.500000,1.480769,1.126923
std,75.199734,1.265680,1.084840
min,1.000000,0.000000,0.000000
25%,65.750000,1.000000,0.000000
50%,130.500000,1.000000,1.000000
75%,195.250000,2.000000,2.000000
max,260.000000,7.000000,5.000000
